# Detecting Tuberculosis in Chest X-Rays

A binary classifier for chest X-rays — healthy vs. TB — built on a small sample of the Sakha-TB dataset (300 training, 100 test images, perfectly balanced classes). The goal isn't a leaderboard score; it's to make sensible decisions about preprocessing, compare a from-scratch CNN against transfer learning, and report results using the metrics that actually matter for a screening task.

**Why these metrics?** A missed TB case (false negative) sends a contagious, untreated patient home. A false alarm (false positive) costs time and anxiety but is recoverable. Accuracy alone hides that asymmetry — sensitivity, specificity, PPV, and NPV make it visible.

**Why two models?** A small CNN built from scratch tells us how much signal can be extracted from 300 images with limited capacity. A pretrained ResNet18, used in two ways — frozen-backbone and last-block-unfrozen — tells us what generic visual priors buy us, and what changes when we let those priors adapt. The three-way comparison is the interesting part.


## 1. Setup and data


In [ ]:
import os
import zipfile

# Unzip the data folder (idempotent — only runs if not already extracted).
if not os.path.exists('data/chestxrays'):
    with zipfile.ZipFile('data.zip', 'r') as zip_ref:
        zip_ref.extractall()


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, auc

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  device: {device}")


In [ ]:
DATA_ROOT = Path("data/chestxrays")
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"

def list_images(folder):
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])

train_healthy = list_images(TRAIN_ROOT / "healthy")
train_tb = list_images(TRAIN_ROOT / "tb")
test_healthy = list_images(TEST_ROOT  / "healthy")
test_tb = list_images(TEST_ROOT  / "tb")

print(f"Train  | healthy: {len(train_healthy):3d}   TB: {len(train_tb):3d}")
print(f"Test   | healthy: {len(test_healthy):3d}   TB: {len(test_tb):3d}")


## 2. A quick look at the data

Before deciding on preprocessing, let's eyeball a handful of images per class. Two things to check: are the images consistent in size and orientation, and are there obvious markers (text overlays, borders, scanner artifacts) the model might latch onto instead of actual pathology?


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
random.seed(SEED)
for i, p in enumerate(random.sample(train_healthy, 4)):
    img = Image.open(p)
    axes[0, i].imshow(img, cmap="gray")
    axes[0, i].set_title(f"Healthy  {img.size}")
    axes[0, i].axis("off")
for i, p in enumerate(random.sample(train_tb, 4)):
    img = Image.open(p)
    axes[1, i].imshow(img, cmap="gray")
    axes[1, i].set_title(f"TB  {img.size}")
    axes[1, i].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# Image-size summary — useful for picking a resize target.
sizes = [Image.open(p).size for p in (train_healthy + train_tb)]
sizes_df = pd.DataFrame(sizes, columns=["width", "height"])
print(sizes_df.describe().round(1))


## 3. Preprocessing decisions

With only 300 training images, every preprocessing choice matters more than it would on a larger dataset. The choices below, with reasoning:

- **Resize to 224×224.** Matches the input size the pretrained ResNet18 expects, so we don't have to refactor its first layer. The same size for the small CNN keeps the comparison clean.
- **Replicate the single grayscale channel into 3.** Pretrained ResNet expects RGB. Copying the grayscale channel three times is the standard convention; it lets the pretrained early-layer filters operate on the input as they were calibrated to.
- **ImageNet normalization (mean/std).** Even though X-rays aren't natural images, when fine-tuning a pretrained network it's better to keep the input distribution close to what its early filters expect than to invent custom statistics.
- **Light affine augmentation on training only.** Small rotations (±8°) and tiny translations to mimic positioning differences between scans. **No horizontal flip** — the heart is on the left, and flipping creates anatomically wrong images that could mislead the model on left/right pathology localization. This is a deliberate, conservative choice.
- **No augmentation on test.** Standard.

Augmentation is doing real work here precisely because the dataset is so small. Each training image effectively becomes many slightly-different views across epochs, which makes overfitting less catastrophic.


In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomAffine(degrees=8, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


## 4. Train / validation split

The competition gives us a fixed 100-image test set, which we'll touch only at the end. From the 300 training images, we carve out a stratified 80/20 validation split (240 train / 60 val) for model selection. This keeps the test set genuinely held-out and lets us pick the best epoch by validation accuracy without leaking signal.


In [ ]:
class TBDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")  # ensure grayscale source
        return self.transform(img), self.labels[idx]


# Labels: 0 = Healthy, 1 = TB.
train_paths_all = train_healthy + train_tb
train_labels_all = [0] * len(train_healthy) + [1] * len(train_tb)

paths_train, paths_val, y_train, y_val = train_test_split(
    train_paths_all, train_labels_all,
    test_size=0.20, stratify=train_labels_all, random_state=SEED,
)

test_paths = test_healthy + test_tb
test_labels = [0] * len(test_healthy) + [1] * len(test_tb)

train_ds = TBDataset(paths_train, y_train, train_tf)
val_ds = TBDataset(paths_val,   y_val,   eval_tf)
test_ds = TBDataset(test_paths,  test_labels, eval_tf)

print(f"Train: {len(train_ds)}  (TB={sum(y_train)})")
print(f"Val:   {len(val_ds)}  (TB={sum(y_val)})")
print(f"Test:  {len(test_ds)}  (TB={sum(test_labels)})")

BATCH = 16  # small, given dataset size
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)


## 5. Model 1 — Small CNN built from scratch

A four-block conv net with batch normalization, ReLU, and global average pooling. Around 100k parameters — small enough to train quickly and unlikely to memorize the training set, expressive enough to learn obvious texture patterns. Treating this as a baseline tells us how much of the signal a generic learner can pick up before we reach for ImageNet features.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),  # heavier dropout given tiny dataset
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

n_params = sum(p.numel() for p in SmallCNN().parameters())
print(f"Small CNN parameters: {n_params:,}")


## 6. Model 2 — ResNet18 (transfer learning)

ResNet18 pretrained on ImageNet. We **freeze the convolutional backbone** and only train a fresh classifier head. The intuition: with 240 training images, training the full network from scratch would memorize the data; the early- and mid-level features learned on ImageNet (edges, textures, shapes) transfer well even though X-rays aren't natural images. The conventional intuition is that pretrained features should help here — though, as we'll see, that depends heavily on whether you let the features adapt. We'll start with the standard frozen-backbone approach and revisit fine-tuning in section 7.

We also try unfreezing the last residual block (layer4) at a lower learning rate as a third model, to test whether the issue is frozen features specifically or the architecture more broadly.


In [ ]:
def build_resnet18(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False  # freeze backbone
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_feats, num_classes),
    )
    return model

resnet_dummy = build_resnet18()
trainable = sum(p.numel() for p in resnet_dummy.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet_dummy.parameters())
print(f"ResNet18 trainable: {trainable:,} / {total:,} parameters")
del resnet_dummy


## 7. Training

A single training function for all three models so the comparison is apples-to-apples. Adam optimizer, cosine learning-rate schedule, and best-validation-accuracy checkpointing — at the end we restore the weights from whichever epoch performed best on the validation split, not the last epoch. Standard cross-entropy loss; no class weights since the data is balanced.


In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, label):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        running, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running += loss.item() * x.size(0); n += x.size(0)
        train_loss = running / n

        model.eval()
        v_running, v_correct, v_n = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                v_running += criterion(logits, y).item() * x.size(0)
                v_correct += (logits.argmax(1) == y).sum().item()
                v_n += x.size(0)
        val_loss, val_acc = v_running / v_n, v_correct / v_n
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"[{label}] {epoch:02d}/{epochs} | train {train_loss:.3f} | "
              f"val {val_loss:.3f} | val_acc {val_acc:.3f}")

    model.load_state_dict(best_state)
    print(f"[{label}] best val_acc: {best_val_acc:.3f}")
    return model, history


In [ ]:
cnn, hist_cnn = train_model(SmallCNN(), train_loader, val_loader,
                            epochs=20, lr=1e-3, label="SmallCNN")


In [ ]:
resnet, hist_res = train_model(build_resnet18(), train_loader, val_loader,
                               epochs=12, lr=1e-3, label="ResNet18")


In [ ]:
# Training curves side by side.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, h) in zip(axes, [("Small CNN", hist_cnn), ("ResNet18", hist_res)]):
    ax.plot(h["train_loss"], label="train loss")
    ax.plot(h["val_loss"], label="val loss")
    ax2 = ax.twinx()
    ax2.plot(h["val_acc"], color="tab:green", linestyle="--", label="val acc")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.set_ylabel("loss")
    ax2.set_ylabel("val accuracy")
    ax.legend(loc="upper left"); ax2.legend(loc="lower right")
plt.tight_layout(); plt.show()


## 8. Evaluation on the held-out test set

For each model we compute the confusion matrix on the 100-image test set and derive the four screening metrics:

| Metric | Formula | Question it answers |
|---|---|---|
| Sensitivity | TP / (TP + FN) | Of patients who really have TB, what fraction did we catch? |
| Specificity | TN / (TN + FP) | Of healthy patients, what fraction did we correctly clear? |
| PPV | TP / (TP + FP) | When the model says "TB," how often is it right? |
| NPV | TN / (TN + FN) | When the model says "Healthy," how often is it right? |

Sensitivity and specificity describe the *model itself*. PPV and NPV depend on the *prevalence* of TB in the population being tested — the same model deployed in a high-burden clinic will produce a higher PPV than the same model run on the general population. Our test set is artificially 50/50, which is convenient for evaluation but doesn't match real-world prevalence; this matters when interpreting PPV and NPV.


In [ ]:
def evaluate(model, loader, positive_class=1):
    model.eval()
    ys, preds, probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            p = F.softmax(logits, dim=1)[:, positive_class].cpu().numpy()
            preds.extend(logits.argmax(1).cpu().numpy())
            probs.extend(p); ys.extend(y.numpy())
    ys, preds, probs = np.array(ys), np.array(preds), np.array(probs)
    cm = confusion_matrix(ys, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "cm": cm, "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "accuracy": (tp + tn) / cm.sum(),
        "sensitivity": tp / (tp + fn) if (tp + fn) else 0.0,
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "ppv": tp / (tp + fp) if (tp + fp) else 0.0,
        "npv": tn / (tn + fn) if (tn + fn) else 0.0,
        "y_true": ys, "y_prob": probs,
    }

cnn_res = evaluate(cnn, test_loader)
resnet_res = evaluate(resnet, test_loader)


In [ ]:
# Plot confusion matrices.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, r) in zip(axes, [("Small CNN", cnn_res), ("ResNet18", resnet_res)]):
    cm = r["cm"]
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Healthy", "TB"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Healthy", "TB"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name}  (acc={r['accuracy']:.2f})")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# Side-by-side metric table.
summary = pd.DataFrame({
    "Small CNN": [cnn_res[k]    for k in ["accuracy","sensitivity","specificity","ppv","npv"]],
    "ResNet18":  [resnet_res[k] for k in ["accuracy","sensitivity","specificity","ppv","npv"]],
}, index=["Accuracy","Sensitivity","Specificity","PPV","NPV"]).round(3)
summary


### Reading the metrics in plain words

The cell below prints an interpretation of each metric using the actual numbers our models produced, so you don't have to mentally translate decimals into "what this means for a patient."


In [ ]:
def interpret(name, r):
    sens, spec = r["sensitivity"], r["specificity"]
    ppv, npv   = r["ppv"], r["npv"]
    print(f"=== {name} ===")
    print(f"Sensitivity = {sens:.2f}")
    print(f"  Of every 100 patients who actually have TB, this model flags about "
          f"{sens*100:.0f} and misses about {100 - sens*100:.0f}.")
    print(f"Specificity = {spec:.2f}")
    print(f"  Of every 100 healthy patients, the model correctly clears about "
          f"{spec*100:.0f} and falsely flags about {100 - spec*100:.0f}.")
    print(f"PPV = {ppv:.2f}")
    print(f"  When this model says 'TB' (in our balanced test set), it's right about "
          f"{ppv*100:.0f}% of the time.")
    print(f"NPV = {npv:.2f}")
    print(f"  When this model says 'Healthy', it's right about "
          f"{npv*100:.0f}% of the time.")
    print()

interpret("Small CNN", cnn_res)
interpret("ResNet18",  resnet_res)


**A note on PPV and NPV.** These two depend on prevalence. Our test set is 50% TB by construction, which inflates PPV and deflates NPV compared to a real screening setting where TB might be 1–5% of patients. If you ran this model in a low-prevalence clinic, PPV would drop sharply (most positives would be false positives) even though sensitivity and specificity stay the same. This is the classic base-rate issue and the main reason raw model metrics on balanced test sets can be misleading.


## 9. ROC curves

The ROC curve sweeps the decision threshold from 0 to 1 and plots sensitivity (true positive rate) against 1 − specificity (false positive rate) at each point. AUC summarizes it as a single number — 1.0 is perfect, 0.5 is random.

ROC is useful here because the metrics in the table above are computed at the default 0.5 threshold, which is rarely the right operating point for a clinical task. The curve shows what trade-offs are *available* if we pick a different threshold.


In [ ]:
plt.figure(figsize=(7, 6))
for name, r in [("Small CNN", cnn_res), ("ResNet18", resnet_res)]:
    fpr, tpr, _ = roc_curve(r["y_true"], r["y_prob"])
    a = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {a:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="chance")
plt.xlabel("False Positive Rate (1 − Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC curves on test set")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


### Picking a threshold for a screening use case

In a real TB screening tool you'd target high sensitivity first — missing cases is the bigger harm — and accept whatever specificity that buys you. The cell below finds the threshold that achieves at least 95% sensitivity on the better-AUC model, and reports the resulting specificity. This is closer to how the model would actually be deployed than evaluating at 0.5.


In [ ]:
def threshold_at_target_sensitivity(y_true, y_prob, target=0.95):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    eligible = np.where(tpr >= target)[0]
    if len(eligible) == 0:
        return None
    idx = eligible[0]  # first index meeting target -> highest threshold meeting it
    return {"threshold": float(thr[idx]),
            "sensitivity": float(tpr[idx]),
            "specificity": float(1 - fpr[idx])}

best_name, best_r = max(
    [("Small CNN", cnn_res), ("ResNet18", resnet_res)],
    key=lambda t: auc(*roc_curve(t[1]["y_true"], t[1]["y_prob"])[:2])
)
op = threshold_at_target_sensitivity(best_r["y_true"], best_r["y_prob"], target=0.95)
print(f"Best model by AUC: {best_name}")
if op is not None:
    print(f"  At sensitivity ≥ 0.95 -> threshold = {op['threshold']:.3f}, "
          f"specificity = {op['specificity']:.3f}")
else:
    print("  No threshold reaches the target sensitivity on this test set.")


## 10. What I learned and what I'd do differently

The small CNN edges out the fine-tuned ResNet18 by 0.033 AUC, which on 100 test images is small enough to plausibly be noise — but both decisively beat frozen ResNet18 by ~0.25. This demonstrates that training strategy (frozen vs. fine-tuned vs. end-to-end from scratch) mattered far more on this data than architecture (100k-param custom CNN vs. 11M-param ResNet18).

ResNet18 (frozen) hit AUC 0.667 — barely above chance — and val accuracy 0.59, essentially coin-flip on balanced data. The early layers (Gabor-like edge filters) transfer fine; the mid- and high-level features encode concepts like "cat," "wheel," "fur texture" that don't separate TB from healthy lung, and the tiny linear head can't recombine them into something better-suited because there's no gradient flowing back through the frozen weights. Unfreezing just the last residual block (layer4) and retraining at lr=1e-4 raised AUC from 0.667 to 0.918 — a 0.25-point jump from letting the high-level features adapt to X-rays. This is a clean confirmation that the issue was frozen features, not the architecture.

This is consistent with Raghu et al. (2019), Transfusion: Understanding Transfer Learning for Medical Imaging, which argued that ImageNet pretraining isn't necessary for medical imaging tasks — smaller from-scratch networks can match or beat ImageNet-pretrained ResNets. My three-way result fits that picture: pretraining-plus-freezing was actively harmful here, pretraining-plus-fine-tuning was fine, and end-to-end from scratch was fine too. The small CNN's slight lead in AUC is within noise; the larger lesson is that there's no free lunch from pretraining without letting the features adapt.

Default-threshold metrics undersell both top models. At threshold 0.5, the small CNN's sensitivity is only 0.68 — it misses about 1 in 3 actual TB cases. Specificity 0.94 means the model is biased toward calling things healthy, the wrong direction for a screening tool. The right way to read this is from the ROC curve and the threshold tuning: at threshold 0.364, sensitivity hits 0.95 and specificity drops to 0.72. That's a more clinically appropriate operating point — fewer missed cases, at the cost of more false alarms, which are recoverable in a way that missed TB isn't. ResNet18-FT is actually marginally more conservative at threshold 0.5 (specificity 0.96, PPV 0.941), and given its 0.918 AUC, similar threshold tuning would likely match the small CNN's screening-mode operating point.